# Unrestricted 317-class cross-dataset evaluation

All 317 output logits are used. The 25 external classes are used only as ground-truth labels; predictions among the other 292 classes remain visible and count as incorrect.


In [2]:
# 1. Imports and centralized configuration
from pathlib import Path
import json, os, time, traceback
import numpy as np, pandas as pd, torch
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
ROOT=Path(r'G:\Ragib\Prescription'); PRIMARY_DATASET_DIR=ROOT/'Full_Dataset_Split'/'Train'; EXTERNAL_DATASET_DIR=ROOT/'Cross_Test_Final'; MODEL_ROOT=ROOT/'Medicine codes'/'Models'
ARCHIVE_ROOT=ROOT/'Medicine Paper Details'/'Codes and Models'/'Codes'; OUTPUT_ROOT=ROOT/'CrossDataset_3Seed_RESULTS'; OUT_DIR=OUTPUT_ROOT/'unrestricted'/'verified_3seed'; os.makedirs(OUT_DIR,exist_ok=True)
PROTOCOL='unrestricted_317_class'; SEEDS=[42,123,456]; MODEL_ORDER=['vgg16','vgg19','densenet121','mobilenet_v2','efficientnet_b0','cnn_bilstm','cnn_attention']; TOTAL_MODELS=len(MODEL_ORDER); TOTAL_SEEDS=len(SEEDS); TOTAL_RUNS=TOTAL_MODELS*TOTAL_SEEDS
NUM_CLASSES=317; NUM_EXTERNAL_CLASSES=25; IMAGE_SIZE=(224,224); BATCH_SIZE=16; NUM_WORKERS=0; RUN_EVALUATION=True; DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OVERLAP_CLASSES=['Ace','Axodin','Beklo','Denixil','Disopan','Esonix','Exium','Fenadin','Fexo','Fexofast','Fixal','Flexibac','Lumona','M-kast','Maxima','Maxpro','Monas','Montene','Napa Extend','Odmon','Renova','Rhinil','Sergel','Trilock','Zithrin']
MODEL_GROUP={'vgg16':'baseline','vgg19':'baseline','densenet121':'baseline','mobilenet_v2':'baseline','efficientnet_b0':'baseline','cnn_bilstm':'bilstm','cnn_attention':'attention'}
ARCHIVE_DIRS={'baseline':ARCHIVE_ROOT/'multirun_baseline','bilstm':ARCHIVE_ROOT/'multirun_bilstm','attention':ARCHIVE_ROOT/'multirun_attention'}
CHECKPOINT_FILENAMES={'vgg16':'best_vgg16.pth','vgg19':'best_vgg19.pth','densenet121':'best_densenet121.pth','mobilenet_v2':'best_mobilenetv2.pth','efficientnet_b0':'best_efficientnetb0.pth','cnn_bilstm':'best_bilstm_317.pth','cnn_attention':'best_attention_317.pth'}
RUN_TO_SEED={1:42,2:123,3:456}; SEED_TO_RUN={seed:run for run,seed in RUN_TO_SEED.items()}; RUN_COMPLETE_MODEL_NAMES={'mobilenet_v2':'mobilenetv2','efficientnet_b0':'efficientnetb0'}
CHECKPOINTS={model:{seed:ARCHIVE_DIRS[MODEL_GROUP[model]]/f'run_{run}'/CHECKPOINT_FILENAMES[model] for run,seed in RUN_TO_SEED.items()} for model in MODEL_ORDER}; RUN_COMPLETE_PATHS={model:{seed:ARCHIVE_DIRS[MODEL_GROUP[model]]/f'run_{run}'/'run_complete.json' for run,seed in RUN_TO_SEED.items()} for model in MODEL_ORDER}
IMAGE_TRANSFORM=transforms.Compose([transforms.Resize(IMAGE_SIZE),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
print(f'Configuration ready | protocol={PROTOCOL} | device={DEVICE} | models={TOTAL_MODELS} | seeds={TOTAL_SEEDS} | runs={TOTAL_RUNS}')


Configuration ready | protocol=unrestricted_317_class | device=cuda | models=7 | seeds=3 | runs=21


In [3]:
# 2. Environment, dataset, preprocessing, and class-mapping validation
required=['os','json','time','traceback','np','pd','torch','matplotlib','plt','sns','OUTPUT_ROOT','OUT_DIR','PROTOCOL','SEEDS','MODEL_ORDER','TOTAL_RUNS','DEVICE','CHECKPOINTS']
missing=[x for x in required if x not in globals()]
if missing: raise RuntimeError('Missing configuration names: '+', '.join(missing))
for label,path in [('primary dataset',PRIMARY_DATASET_DIR),('external dataset',EXTERNAL_DATASET_DIR),('archive root',ARCHIVE_ROOT)]:
    if not path.is_dir(): raise FileNotFoundError(f'{label} does not exist: {path}')
primary_ds=datasets.ImageFolder(PRIMARY_DATASET_DIR); external_ds=datasets.ImageFolder(EXTERNAL_DATASET_DIR,transform=IMAGE_TRANSFORM)
primary_classes=list(primary_ds.classes); external_classes=list(external_ds.classes); primary_to_global=primary_ds.class_to_idx
if len(primary_classes)!=NUM_CLASSES: raise ValueError(f'Expected {NUM_CLASSES} primary classes, found {len(primary_classes)}')
if set(external_classes)!=set(OVERLAP_CLASSES) or len(external_classes)!=NUM_EXTERNAL_CLASSES: raise ValueError('External classes do not exactly match the expected 25-class vocabulary.')
if not set(external_classes).issubset(primary_classes): raise ValueError('At least one external class is missing from the primary vocabulary.')
external_global_ids=np.asarray([primary_to_global[name] for name in external_classes],dtype=np.int64); external_global_to_local={int(g):i for i,g in enumerate(external_global_ids)}
counts=np.bincount(np.asarray(external_ds.targets),minlength=NUM_EXTERNAL_CLASSES)
if len(external_ds)!=250 or not np.all(counts==10): raise ValueError(f'Expected 250 images and 10/class; found {len(external_ds)} and {counts.tolist()}')
loader=DataLoader(external_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')
if len(loader)==0: raise RuntimeError('External DataLoader is empty.')
print(f'Validated primary={len(primary_classes)} classes | external={len(external_classes)} classes | images={len(external_ds)} | batches={len(loader)} | preprocessing=Resize{IMAGE_SIZE}+ImageNet normalization')


Validated primary=317 classes | external=25 classes | images=250 | batches=16 | preprocessing=Resize(224, 224)+ImageNet normalization


In [4]:
# 3. Original model architectures
class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__(); self.mobilenet=models.mobilenet_v2(weights=None); self.mobilenet.classifier=nn.Identity(); self.efficientnet=models.efficientnet_b0(weights=None); self.efficientnet.classifier=nn.Identity(); self.lstm=nn.LSTM(2560,256,batch_first=True,bidirectional=True); self.classifier=nn.Sequential(nn.Linear(512,256),nn.ReLU(),nn.Dropout(.5),nn.Linear(256,NUM_CLASSES))
    def forward(self,x):
        z=torch.cat([self.mobilenet(x),self.efficientnet(x)],1).unsqueeze(1); z,_=self.lstm(z); return self.classifier(z[:,-1])
class Attention(nn.Module):
    def __init__(self):
        super().__init__(); self.attn=nn.Sequential(nn.Linear(2560,1280),nn.ReLU(),nn.Linear(1280,2),nn.Softmax(1))
    def forward(self,a,b):
        w=self.attn(torch.cat([a,b],1)); return w[:,:1]*a+w[:,1:2]*b
class CNNAttention(nn.Module):
    def __init__(self):
        super().__init__(); self.mobilenet=models.mobilenet_v2(weights=None); self.mobilenet.classifier=nn.Identity(); self.efficientnet=models.efficientnet_b0(weights=None); self.efficientnet.classifier=nn.Identity(); self.fusion=Attention(); self.classifier=nn.Sequential(nn.Linear(1280,256),nn.ReLU(),nn.Dropout(.5),nn.Linear(256,NUM_CLASSES))
    def forward(self,x): return self.classifier(self.fusion(self.mobilenet(x),self.efficientnet(x)))
def build_model(name):
    if name=='vgg16': m=models.vgg16(weights=None); m.classifier[6]=nn.Linear(4096,NUM_CLASSES)
    elif name=='vgg19': m=models.vgg19(weights=None); m.classifier[6]=nn.Linear(4096,NUM_CLASSES)
    elif name=='densenet121': m=models.densenet121(weights=None); m.classifier=nn.Linear(m.classifier.in_features,NUM_CLASSES)
    elif name=='mobilenet_v2': m=models.mobilenet_v2(weights=None); m.classifier[1]=nn.Linear(m.classifier[1].in_features,NUM_CLASSES)
    elif name=='efficientnet_b0': m=models.efficientnet_b0(weights=None); m.classifier[1]=nn.Linear(m.classifier[1].in_features,NUM_CLASSES)
    elif name=='cnn_bilstm': m=BiLSTM()
    elif name=='cnn_attention': m=CNNAttention()
    else: raise ValueError(f'Unknown model: {name}')
    return m
def unwrap_state_dict(x):
    if isinstance(x,dict):
        for key in ('model','model_state','model_state_dict','state_dict'):
            if isinstance(x.get(key),dict): return x[key]
        if x and all(isinstance(v,torch.Tensor) for v in x.values()): return x
    raise TypeError('Unsupported checkpoint wrapper.')
def normalized_state_dict(path):
    raw=torch.load(path,map_location='cpu',weights_only=False); return {k.removeprefix('module.'):v for k,v in unwrap_state_dict(raw).items()}


In [5]:
# 4. Checkpoint preflight: paths, run metadata, architecture compatibility
def load_model(name,path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f'Checkpoint does not exist: {path}')
    model=build_model(name); state=normalized_state_dict(path); missing_keys,unexpected_keys=model.load_state_dict(state,strict=False)
    if missing_keys or unexpected_keys: raise RuntimeError(f'Checkpoint key mismatch for {name}: missing={missing_keys}, unexpected={unexpected_keys}')
    return model.to(DEVICE).eval()
def read_run_record(path,metadata_name):
    data=json.loads(path.read_text(encoding='utf-8')); records=data if isinstance(data,list) else [data]; matches=[record for record in records if not isinstance(data,list) or record.get('Model',metadata_name)==metadata_name]
    if len(matches)!=1: raise RuntimeError(f'Expected exactly one {metadata_name} record in {path}, found {len(matches)}')
    return matches[0]
available_runs=[]
for name in MODEL_ORDER:
    for seed in SEEDS:
        checkpoint=CHECKPOINTS[name][seed]; complete=RUN_COMPLETE_PATHS[name][seed]
        if not checkpoint.is_file(): raise FileNotFoundError(f'MISSING checkpoint: model={name}, seed={seed}, path={checkpoint}')
        if not complete.is_file(): raise FileNotFoundError(f'MISSING run_complete.json: model={name}, seed={seed}, path={complete}')
        record=read_run_record(complete,RUN_COMPLETE_MODEL_NAMES.get(name,name)); recorded_seed=record.get('Random Seed')
        if recorded_seed!=seed: raise RuntimeError(f'SEED MISMATCH before evaluation: model={name}, expected={seed}, recorded={recorded_seed}, metadata={complete}')
        probe=load_model(name,checkpoint); del probe
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
        run_number=SEED_TO_RUN[seed]; available_runs.append({'model':name,'seed':seed,'run_number':run_number,'checkpoint':str(checkpoint),'run_complete':str(complete),'verification':'verified'})
        print(f'VALIDATED model={name} | run_{run_number} | seed={seed} | checkpoint={checkpoint}',flush=True)
if len(available_runs)!=TOTAL_RUNS: raise RuntimeError(f'Expected {TOTAL_RUNS} verified runs, found {len(available_runs)}')
print(f'All {len(available_runs)}/{TOTAL_RUNS} checkpoint paths, seeds, and architecture loads validated before inference.')


VALIDATED model=vgg16 | run_1 | seed=42 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_1\best_vgg16.pth
VALIDATED model=vgg16 | run_2 | seed=123 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_2\best_vgg16.pth
VALIDATED model=vgg16 | run_3 | seed=456 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_3\best_vgg16.pth
VALIDATED model=vgg19 | run_1 | seed=42 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_1\best_vgg19.pth
VALIDATED model=vgg19 | run_2 | seed=123 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_2\best_vgg19.pth
VALIDATED model=vgg19 | run_3 | seed=456 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_3\best_vgg19.pth
VALIDATED model=densenet121 | run_1 

In [7]:
# 5. Unrestricted metrics: all 317 logits; 25 supported ground-truth labels
def evaluate_model(model,run):
    y_true=[]; y_pred_global=[]; top5_global=[]; started=time.perf_counter(); total_batches=len(loader)
    print(f'Protocol={PROTOCOL} | device={DEVICE} | model={run["model"]} | seed={run["seed"]} | checkpoint={run["checkpoint"]}',flush=True)
    with torch.inference_mode():
        for batch_number,(images,labels) in enumerate(loader,1):
            logits=model(images.to(DEVICE,non_blocking=True))
            if logits.ndim!=2 or logits.shape[1]!=NUM_CLASSES: raise ValueError(f'Expected complete [batch,317] logits, got {tuple(logits.shape)}')
            y_true.extend(labels.numpy().tolist()); y_pred_global.extend(logits.argmax(1).cpu().numpy().tolist()); top5_global.extend(torch.topk(logits,5,1).indices.cpu().numpy().tolist())
            print(f'Batch {batch_number}/{total_batches} | elapsed={time.perf_counter()-started:.1f}s',flush=True)
    y_true=np.asarray(y_true,dtype=np.int64); y_pred_global=np.asarray(y_pred_global,dtype=np.int64); y_pred_local=np.asarray([external_global_to_local.get(int(x),-1) for x in y_pred_global],dtype=np.int64); labels25=np.arange(NUM_EXTERNAL_CLASSES)
    top5_hit=np.asarray([int(int(external_global_ids[t]) in ranked) for t,ranked in zip(y_true,top5_global)],dtype=np.int64); outside_count=int(np.sum(y_pred_local<0)); outside_rate=float(outside_count/len(y_true))
    macro=precision_recall_fscore_support(y_true,y_pred_local,labels=labels25,average='macro',zero_division=0); weighted=precision_recall_fscore_support(y_true,y_pred_local,labels=labels25,average='weighted',zero_division=0); micro=precision_recall_fscore_support(y_true,y_pred_local,labels=labels25,average='micro',zero_division=0)
    report=classification_report(y_true,y_pred_local,labels=labels25,target_names=external_classes,zero_division=0,output_dict=True); cm=np.zeros((NUM_EXTERNAL_CLASSES,NUM_CLASSES),dtype=np.int64); np.add.at(cm,(y_true,y_pred_global),1)
    if cm.shape!=(25,317): raise RuntimeError(f'Confusion matrix shape is {cm.shape}, expected (25,317)')
    return {'y_true':y_true,'y_pred_global':y_pred_global,'y_pred_local':y_pred_local,'top5_global':top5_global,'top1_accuracy':float(accuracy_score(y_true,y_pred_local)),'top5_accuracy':float(top5_hit.mean()),'precision':float(micro[0]),'recall':float(micro[1]),'f1_score':float(micro[2]),'macro_precision':float(macro[0]),'macro_recall':float(macro[1]),'macro_f1':float(macro[2]),'weighted_precision':float(weighted[0]),'weighted_recall':float(weighted[1]),'weighted_f1':float(weighted[2]),'outside_292_predictions':outside_count,'outside_292_rate':outside_rate,'report':report,'cm':cm,'duration_seconds':time.perf_counter()-started}
def save_full_cm(cm,title,path):
    plt.figure(figsize=(80,14)); sns.heatmap(cm,cmap='Blues',xticklabels=primary_classes,yticklabels=external_classes,cbar=True); plt.title(title); plt.xlabel('Predicted original 317-class index/name'); plt.ylabel('True external class'); plt.xticks(rotation=90,fontsize=4); plt.yticks(rotation=0,fontsize=7); plt.tight_layout(); plt.savefig(path,dpi=200,bbox_inches='tight'); plt.close()


In [8]:
# 6. Main 21-run unrestricted evaluation with progressive saving
metrics=['top1_accuracy','top5_accuracy','precision','recall','f1_score','macro_precision','macro_recall','macro_f1','weighted_precision','weighted_recall','weighted_f1','outside_292_predictions','outside_292_rate']; rows=[]; matrices={name:[] for name in MODEL_ORDER}; overall_started=time.perf_counter()
print(f'Protocol: {PROTOCOL} | Device: {DEVICE} | Models: {TOTAL_MODELS} | Seeds: {TOTAL_SEEDS} | Total runs: {TOTAL_RUNS}',flush=True)
for run_number,run in enumerate(available_runs,1):
    name=run['model']; seed=run['seed']; model=None; run_started=time.perf_counter()
    try:
        print(f'\n[{run_number}/{TOTAL_RUNS}] Current model={name} | seed={seed} | checkpoint={run["checkpoint"]}',flush=True); model=load_model(name,run['checkpoint']); result=evaluate_model(model,run); run_dir=OUT_DIR/name/f'seed_{seed}'; run_dir.mkdir(parents=True,exist_ok=True)
        prediction_df=pd.DataFrame({'true_class':[external_classes[i] for i in result['y_true']],'true_primary_index':external_global_ids[result['y_true']],'predicted_class':[primary_classes[i] for i in result['y_pred_global']],'predicted_primary_index':result['y_pred_global'],'predicted_external_index':result['y_pred_local'],'top5_primary_classes':['|'.join(primary_classes[i] for i in ranked) for ranked in result['top5_global']]}); prediction_df.to_csv(run_dir/'predictions.csv',index=False)
        (run_dir/'classification_report.json').write_text(json.dumps(result['report'],indent=2),encoding='utf-8'); pd.DataFrame(result['report']).T.to_csv(run_dir/'classification_report.csv',index_label='class'); pd.DataFrame(result['cm'],index=external_classes,columns=primary_classes).to_csv(run_dir/'confusion_matrix_25x317.csv'); save_full_cm(result['cm'],f'{name.upper()} | UNRESTRICTED 317-CLASS | Seed {seed}',run_dir/'confusion_matrix_25x317.png')
        row={'model':name,'seed':seed,'run_number':run['run_number'],'checkpoint':run['checkpoint'],'checkpoint_verification':run['verification'],'evaluation_duration_seconds':result['duration_seconds'],**{metric:result[metric] for metric in metrics}}; rows.append(row); matrices[name].append(result['cm']); pd.DataFrame(rows).to_csv(OUT_DIR/'completed_runs_progressive.csv',index=False)
        print(f'Run completed: model={name} | seed={seed} | duration={(time.perf_counter()-run_started):.1f}s | overall={len(rows)}/{TOTAL_RUNS}',flush=True)
    except Exception as exc:
        print(f'RUN FAILED | model={name} | seed={seed} | checkpoint={run["checkpoint"]} | {type(exc).__name__}: {exc}',flush=True); traceback.print_exc()
    finally:
        if model is not None: del model
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
print(f'Overall completion: {len(rows)}/{TOTAL_RUNS} runs in {(time.perf_counter()-overall_started)/60:.2f} minutes',flush=True)
if len(rows)!=TOTAL_RUNS: raise RuntimeError(f'Not all runs completed: {len(rows)}/{TOTAL_RUNS}')


Protocol: unrestricted_317_class | Device: cuda | Models: 7 | Seeds: 3 | Total runs: 21

[1/21] Current model=vgg16 | seed=42 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_1\best_vgg16.pth
Protocol=unrestricted_317_class | device=cuda | model=vgg16 | seed=42 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_1\best_vgg16.pth
Batch 1/16 | elapsed=0.9s
Batch 2/16 | elapsed=1.0s
Batch 3/16 | elapsed=1.1s
Batch 4/16 | elapsed=1.1s
Batch 5/16 | elapsed=1.2s
Batch 6/16 | elapsed=1.3s
Batch 7/16 | elapsed=1.4s
Batch 8/16 | elapsed=1.4s
Batch 9/16 | elapsed=1.5s
Batch 10/16 | elapsed=1.6s
Batch 11/16 | elapsed=1.7s
Batch 12/16 | elapsed=1.7s
Batch 13/16 | elapsed=1.8s
Batch 14/16 | elapsed=1.9s
Batch 15/16 | elapsed=2.0s
Batch 16/16 | elapsed=2.0s
Run completed: model=vgg16 | seed=42 | duration=5.9s | overall=1/21

[2/21] Current model=vgg16 | seed=123 | checkpoint=G:\Ragib\Prescription

In [9]:
# 7. Per-seed export, mean/sample-SD summaries, and aggregate confusion matrices
per_run=pd.DataFrame(rows); per_run.to_csv(OUT_DIR/'all_models_all_seeds_results.csv',index=False); summary=[]
for name in MODEL_ORDER:
    subset=per_run[per_run.model==name]
    if len(subset)!=3: raise RuntimeError(f'{name} has {len(subset)} completed seeds; expected 3')
    summary_row={'model':name,'n_seeds':len(subset),'seeds':'42,123,456'}
    for metric in metrics:
        values=subset[metric].astype(float).to_numpy(); mean=float(values.mean()); sd=float(values.std(ddof=1)); summary_row[metric+'_mean']=mean; summary_row[metric+'_sample_sd']=sd; summary_row[metric+'_mean_plus_minus_sd']=f'{mean:.6f} +/- {sd:.6f}'
    summary.append(summary_row)
summary_df=pd.DataFrame(summary); summary_df.to_csv(OUT_DIR/'summary_mean_sample_sd_3seeds.csv',index=False)
for name in MODEL_ORDER:
    stacked=np.stack(matrices[name]); model_dir=OUT_DIR/name; model_dir.mkdir(parents=True,exist_ok=True)
    if stacked.shape!=(3,25,317): raise RuntimeError(f'{name} aggregate confusion shape is {stacked.shape}, expected (3,25,317)')
    pd.DataFrame(stacked.sum(axis=0),index=external_classes,columns=primary_classes).to_csv(model_dir/'confusion_matrix_sum_3seeds_25x317.csv'); pd.DataFrame(stacked.mean(axis=0),index=external_classes,columns=primary_classes).to_csv(model_dir/'confusion_matrix_mean_3seeds_25x317.csv')
print(f'Saved 21 per-seed runs, 7 summaries, and 25x317 confusion matrices under {OUT_DIR}')
print('Final status: unrestricted 317-class evaluation completed successfully.')


Saved 21 per-seed runs, 7 summaries, and 25x317 confusion matrices under G:\Ragib\Prescription\CrossDataset_3Seed_RESULTS\unrestricted\verified_3seed
Final status: unrestricted 317-class evaluation completed successfully.
